# MLP Experiment: RGB + Depth + Joint Encoders

## Objective
This notebook implements a **multimodal gated fusion model** for posture classification using three pretrained encoders:

- **RGB encoder**
- **Depth encoder**
- **Joint encoder**

The goal is to combine feature representations from all three modalities and train a **gated fusion classification head** on top of them.

---

## Experiment Setup
In this experiment:

- pretrained encoder weights are loaded from saved artifacts
- encoder backbones are used as **feature extractors**
- encoder parameters are initially **frozen**
- a **gated fusion module** is trained to combine modality-specific features
- the final classifier predicts one of the posture classes:
  - **supine**
  - **left**
  - **right**

---

## Why Gated Fusion
Gated fusion allows the model to learn the **relative importance of each modality** for a given sample.

This is useful because:

- **RGB** may become less reliable under blanket occlusion
- **Depth** may retain useful spatial posture information
- **Joint features** may provide structural cues but may also be noisy

The gating mechanism helps the model dynamically weight each modality before final classification.

## Imports and Setup

In [1]:
# =========================
# Core
# =========================
from pathlib import Path

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

In [2]:
# =========================
# Device setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Paths and Configuration

In [3]:
# =========================
# Paths
# =========================
PROJECT_ROOT = Path.cwd().resolve().parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

DEPTH_ARTIFACT_DIR = ARTIFACTS_DIR / "depth_encoder_cnn" / "checkpoints"
RGB_ARTIFACT_DIR = ARTIFACTS_DIR / "rgb_encoder_cnn" / "checkpoints"
JOINT_ARTIFACT_DIR = ARTIFACTS_DIR / "joint_xyo" / "checkpoints"

FUSION_CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints_fusion_gated"
FUSION_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Depth artifacts:", DEPTH_ARTIFACT_DIR)
print("RGB artifacts:", RGB_ARTIFACT_DIR)
print("Joint artifacts:", JOINT_ARTIFACT_DIR)
print("Fusion checkpoints:", FUSION_CHECKPOINT_DIR)

Project root: C:\Users\Fia Thottan\ApneaSense
Depth artifacts: C:\Users\Fia Thottan\ApneaSense\artifacts\depth_encoder_cnn\checkpoints
RGB artifacts: C:\Users\Fia Thottan\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints
Joint artifacts: C:\Users\Fia Thottan\ApneaSense\artifacts\joint_xyo\checkpoints
Fusion checkpoints: C:\Users\Fia Thottan\ApneaSense\artifacts\checkpoints_fusion_gated


## Experiment Configuration

Define the basic experiment settings for gated fusion, including:

- number of classes
- batch size
- learning rate
- number of epochs
- common feature dimension for fusion

These can be adjusted later if needed.

In [4]:
# =========================
# Experiment config
# =========================
NUM_CLASSES = 3

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15

COMMON_FEATURE_DIM = 256 # to convert all features from encoders to a common size
DROPOUT = 0.3

CLASS_NAMES = ["supine", "left", "right"]

## Encoder Checkpoint Paths

In [5]:
# =========================
# Encoder checkpoint files
# =========================
DEPTH_CHECKPOINT_PATH = DEPTH_ARTIFACT_DIR / "best_depth_encoder_only.pth"
RGB_CHECKPOINT_PATH = RGB_ARTIFACT_DIR / "best_rgb_encoder.pt"
JOINT_CHECKPOINT_PATH = JOINT_ARTIFACT_DIR / "joint_encoder_xyo_RGB.pth"

print("Depth checkpoint:", DEPTH_CHECKPOINT_PATH)
print("RGB checkpoint:", RGB_CHECKPOINT_PATH)
print("Joint checkpoint:", JOINT_CHECKPOINT_PATH)

Depth checkpoint: C:\Users\Fia Thottan\ApneaSense\artifacts\depth_encoder_cnn\checkpoints\best_depth_encoder_only.pth
RGB checkpoint: C:\Users\Fia Thottan\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints\best_rgb_encoder.pt
Joint checkpoint: C:\Users\Fia Thottan\ApneaSense\artifacts\joint_xyo\checkpoints\joint_encoder_xyo_RGB.pth


## Encoder Model Definitions

Define the encoder architectures needed to load the pretrained weights.

At this stage, we only recreate the model structures required for:

- RGB encoder
- Depth encoder
- Joint encoder

These definitions should match the architectures used during the original encoder training notebooks.

In [6]:
# =========================
# Encoder feature dimensions
# =========================
DEPTH_FEATURE_DIM = 512
RGB_FEATURE_DIM = 128
JOINT_INPUT_DIM = 42
JOINT_FEATURE_DIM = 128

In [7]:
class DepthEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)

        # change first conv to 1 channel
        backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # remove FC
        layers = list(backbone.children())[:-1]

        self.encoder = nn.Sequential(*layers)
        self.flatten = nn.Flatten()

        self.feature_dim = 512

    def forward(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        return x


class RGBEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128)
        )
        self.feature_dim = 128

    def forward(self, x):
        x = self.backbone(x)
        x = self.projection(x)
        return x


class JointEncoder(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=128, feature_dim=128, dropout=0.3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.feature_dim = feature_dim

    def forward(self, x):
        return self.encoder(x)

## Load Pretrained Encoders

Instantiate the three encoder models and load their encoder-only weights from the saved checkpoints.

After loading:
- move models to the selected device
- set them to evaluation mode
- freeze parameters so that only the gated fusion head is trained in this first experiment stage

In [8]:
# =========================
# Instantiate encoders
# =========================
depth_encoder = DepthEncoder().to(device)
rgb_encoder = RGBEncoder().to(device)
joint_encoder = JointEncoder(
    input_dim=JOINT_INPUT_DIM,
    hidden_dim=128,
    feature_dim=JOINT_FEATURE_DIM,
    dropout=0.3
).to(device)

# =========================
# Load checkpoints
# =========================
depth_ckpt = torch.load(DEPTH_CHECKPOINT_PATH, map_location=device)
rgb_ckpt = torch.load(RGB_CHECKPOINT_PATH, map_location=device)
joint_ckpt = torch.load(JOINT_CHECKPOINT_PATH, map_location=device)

depth_encoder.encoder.load_state_dict(depth_ckpt["encoder_state_dict"])
rgb_encoder.load_state_dict(rgb_ckpt["encoder_state_dict"])
joint_encoder.encoder.load_state_dict(joint_ckpt["encoder_state_dict"])

# =========================
# Freeze encoders
# =========================
for model in [depth_encoder, rgb_encoder, joint_encoder]:
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("All encoders loaded and frozen successfully.")

All encoders loaded and frozen successfully.


# MLP Model

In [9]:
import torch
import torch.nn as nn

class ConcatFusionModel(nn.Module):
    def __init__(
        self,
        depth_encoder,
        rgb_encoder,
        joint_encoder,
        depth_feature_dim=512,
        rgb_feature_dim=128,
        joint_feature_dim=128,
        common_feature_dim=256,
        num_classes=3,
        dropout=0.3
    ):
        super().__init__()

        self.depth_encoder = depth_encoder
        self.rgb_encoder = rgb_encoder
        self.joint_encoder = joint_encoder

        # projection layers
        self.depth_proj = nn.Sequential(
            nn.Linear(depth_feature_dim, common_feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.rgb_proj = nn.Sequential(
            nn.Linear(rgb_feature_dim, common_feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.joint_proj = nn.Sequential(
            nn.Linear(joint_feature_dim, common_feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(common_feature_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, depth, rgb, joint):
        with torch.no_grad():
            f_depth = self.depth_encoder(depth)
            f_rgb   = self.rgb_encoder(rgb)
            f_joint = self.joint_encoder(joint)

        f_depth = self.depth_proj(f_depth)
        f_rgb   = self.rgb_proj(f_rgb)
        f_joint = self.joint_proj(f_joint)

        fused = torch.cat([f_depth, f_rgb, f_joint], dim=1)
        out = self.classifier(fused)

        return out

In [10]:
model = ConcatFusionModel(
    depth_encoder=depth_encoder,
    rgb_encoder=rgb_encoder,
    joint_encoder=joint_encoder,
    depth_feature_dim=DEPTH_FEATURE_DIM,
    rgb_feature_dim=RGB_FEATURE_DIM,
    joint_feature_dim=JOINT_FEATURE_DIM,
    common_feature_dim=COMMON_FEATURE_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

print(model)

ConcatFusionModel(
  (depth_encoder): DepthEncoder(
    (encoder): Sequential(
      (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): Batc

In [11]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
)

In [14]:
#sanity check -test
model.eval()

# fake batch
depth = torch.randn(32, 1, 224, 224).to(device)   # depth image
rgb   = torch.randn(32, 3, 224, 224).to(device)   # rgb image
joint = torch.randn(32, 42).to(device)            # joint input

with torch.no_grad():
    output = model(depth, rgb, joint)

print("Output shape:", output.shape)

Output shape: torch.Size([32, 3])


In [21]:
import pandas as pd

csv_path = r"C:\Users\Fia Thottan\SensingProject\SLP2022\SLP\danaLab\posture_labels_all_modalities.csv"
df = pd.read_csv(csv_path)

print(df.head())
print(df.columns.tolist())
print(df["modality"].unique())
print(df.shape)

   subject_id modality condition  image_index  \
0           1      RGB   uncover            1   
1           1      RGB   uncover            2   
2           1      RGB   uncover            3   
3           1      RGB   uncover            4   
4           1      RGB   uncover            5   

                           image_path   label  label_id             source  
0  00001/RGB/uncover/image_000001.png  supine         0  inferred_ordering  
1  00001/RGB/uncover/image_000002.png  supine         0  inferred_ordering  
2  00001/RGB/uncover/image_000003.png  supine         0  inferred_ordering  
3  00001/RGB/uncover/image_000004.png  supine         0  inferred_ordering  
4  00001/RGB/uncover/image_000005.png  supine         0  inferred_ordering  
['subject_id', 'modality', 'condition', 'image_index', 'image_path', 'label', 'label_id', 'source']
['RGB' 'IR' 'depth' 'PM']
(55080, 8)


In [22]:
df = pd.read_csv(csv_path)

print(df.columns.tolist())
print(df["modality"].unique())

['subject_id', 'modality', 'condition', 'image_index', 'image_path', 'label', 'label_id', 'source']
['RGB' 'IR' 'depth' 'PM']


In [23]:
# =========================
# Dataset + DataLoader for Concat Fusion
# =========================

from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# -------------------------------------------------
# IMPORTANT: set this to YOUR local danaLab folder
# -------------------------------------------------
DATASET_ROOT = Path(r"C:\Users\Fia Thottan\SensingProject\SLP2022\SLP\danaLab")
CSV_PATH = DATASET_ROOT / "posture_labels_all_modalities.csv"

print("DATASET_ROOT:", DATASET_ROOT)
print("CSV_PATH:", CSV_PATH)
print("CSV exists:", CSV_PATH.exists())

# =========================
# Build aligned metadata
# =========================
def build_fusion_metadata(csv_path):
    df = pd.read_csv(csv_path)
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)

    # Use RGB rows as the base sample definition
    rgb_df = df[df["modality"] == "RGB"].copy().reset_index(drop=True)

    rgb_df["rgb_path"] = (
        rgb_df["subject_id"] + "/RGB/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )

    rgb_df["depth_path"] = (
        rgb_df["subject_id"] + "/depth/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )

    # Joints are stored once per subject
    rgb_df["joint_file"] = rgb_df["subject_id"] + "/joints_gt_RGB.mat"
    rgb_df["frame_idx_0based"] = rgb_df["image_index"].astype(int) - 1

    return rgb_df[[
        "subject_id",
        "condition",
        "image_index",
        "label",
        "label_id",
        "rgb_path",
        "depth_path",
        "joint_file",
        "frame_idx_0based",
    ]].reset_index(drop=True)

# =========================
# Joint preprocessing
# =========================
IMG_W = 576.0
IMG_H = 1024.0

def preprocess_joint_frame_xyo(frame_joints):
    """
    frame_joints shape expected like [3, num_joints]
    row 0 = x, row 1 = y, row 2 = occupancy/visibility
    output = flattened 42-dim vector (14 joints * 3 values)
    """
    x = frame_joints[0].astype(np.float32) / IMG_W
    y = frame_joints[1].astype(np.float32) / IMG_H
    occ = frame_joints[2].astype(np.float32)

    out = np.stack([x, y, occ], axis=1)   # [num_joints, 3]
    out = out.reshape(-1)                 # [42]

    return out

# =========================
# Image transforms
# =========================
rgb_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

depth_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

# =========================
# Fusion dataset
# =========================
class FusionDataset(Dataset):
    def __init__(self, df, dataset_root, rgb_transform=None, depth_transform=None):
        self.df = df.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.rgb_transform = rgb_transform
        self.depth_transform = depth_transform
        self.joint_cache = {}

    def __len__(self):
        return len(self.df)

    def _load_joint_file(self, joint_rel_path):
        if joint_rel_path not in self.joint_cache:
            full_joint_path = self.dataset_root / joint_rel_path
            mat = sio.loadmat(full_joint_path)
            self.joint_cache[joint_rel_path] = mat["joints_gt"]
        return self.joint_cache[joint_rel_path]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # RGB
        rgb_path = self.dataset_root / row["rgb_path"]
        rgb_img = Image.open(rgb_path).convert("RGB")
        if self.rgb_transform is not None:
            rgb_img = self.rgb_transform(rgb_img)

        # Depth
        depth_path = self.dataset_root / row["depth_path"]
        depth_img = Image.open(depth_path).convert("L")
        if self.depth_transform is not None:
            depth_img = self.depth_transform(depth_img)

        # Joint vector
        joints_all = self._load_joint_file(row["joint_file"])
        frame = joints_all[:, :, int(row["frame_idx_0based"])]
        joint_vec = preprocess_joint_frame_xyo(frame)

        return {
            "rgb": rgb_img,
            "depth": depth_img,
            "joint": torch.tensor(joint_vec, dtype=torch.float32),
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "condition": row["condition"],
            "subject_id": row["subject_id"],
        }

# =========================
# Subject-wise split
# =========================
def subject_wise_split(subject_ids, train_ratio=0.70, val_ratio=0.15, seed=42):
    subject_ids = sorted(subject_ids)
    rng = random.Random(seed)
    rng.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_subjects = subject_ids[:n_train]
    val_subjects = subject_ids[n_train:n_train + n_val]
    test_subjects = subject_ids[n_train + n_val:]

    return train_subjects, val_subjects, test_subjects

# =========================
# Build metadata and loaders
# =========================
fusion_df = build_fusion_metadata(CSV_PATH)

subjects = sorted(fusion_df["subject_id"].unique().tolist())
train_subjects, val_subjects, test_subjects = subject_wise_split(
    subjects, train_ratio=0.70, val_ratio=0.15, seed=42
)

train_df = fusion_df[fusion_df["subject_id"].isin(train_subjects)].reset_index(drop=True)
val_df   = fusion_df[fusion_df["subject_id"].isin(val_subjects)].reset_index(drop=True)
test_df  = fusion_df[fusion_df["subject_id"].isin(test_subjects)].reset_index(drop=True)

train_dataset = FusionDataset(train_df, DATASET_ROOT, rgb_transform, depth_transform)
val_dataset   = FusionDataset(val_df, DATASET_ROOT, rgb_transform, depth_transform)
test_dataset  = FusionDataset(test_df, DATASET_ROOT, rgb_transform, depth_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Fusion metadata shape:", fusion_df.shape)
print("Train / Val / Test sizes:", len(train_dataset), len(val_dataset), len(test_dataset))

DATASET_ROOT: C:\Users\Fia Thottan\SensingProject\SLP2022\SLP\danaLab
CSV_PATH: C:\Users\Fia Thottan\SensingProject\SLP2022\SLP\danaLab\posture_labels_all_modalities.csv
CSV exists: True
Fusion metadata shape: (13770, 9)
Train / Val / Test sizes: 9585 2025 2160


In [25]:

model.eval()

batch = next(iter(train_loader))

depth = batch["depth"].to(device)
rgb   = batch["rgb"].to(device)
joint = batch["joint"].to(device)

with torch.no_grad():
    output = model(depth, rgb, joint)

print("Output shape:", output.shape)

Output shape: torch.Size([32, 3])


In [26]:

# Loss function
# Since this is a 3-class classification problem
# (supine / left / right), CrossEntropyLoss is appropriate.
criterion = nn.CrossEntropyLoss()

In [27]:
from sklearn.metrics import f1_score


# Train for one epoch

def train_one_epoch(model, loader, criterion, optimizer, device):
    # Put model into training mode
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    # Loop through one full pass of the training data
    for batch in loader:
        # Move batch tensors to GPU/CPU
        depth = batch["depth"].to(device)
        rgb   = batch["rgb"].to(device)
        joint = batch["joint"].to(device)
        labels = batch["label"].to(device)

        # Clear gradients from previous step
        optimizer.zero_grad()

        # Forward pass
        outputs = model(depth, rgb, joint)

        # Compute classification loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update trainable parameters
        optimizer.step()

        # Accumulate total loss
        running_loss += loss.item() * labels.size(0)

        # Get predicted class index
        preds = outputs.argmax(dim=1)

        # Count correct predictions
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    # Average loss and accuracy across epoch
    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [28]:

# Evaluate model on validation or test set

def evaluate(model, loader, criterion, device):
    # Put model into evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # Store predictions and true labels for F1 / confusion matrix later
    all_preds = []
    all_labels = []

    # No gradient tracking during evaluation
    with torch.no_grad():
        for batch in loader:
            # Move batch tensors to GPU/CPU
            depth = batch["depth"].to(device)
            rgb   = batch["rgb"].to(device)
            joint = batch["joint"].to(device)
            labels = batch["label"].to(device)

            # Forward pass
            outputs = model(depth, rgb, joint)

            # Compute loss
            loss = criterion(outputs, labels)

            # Add to running total
            running_loss += loss.item() * labels.size(0)

            # Predicted class
            preds = outputs.argmax(dim=1)

            # Accuracy counting
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Save predictions and labels for later metrics
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute overall metrics
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, macro_f1, all_preds, all_labels

In [29]:

# Training loop


# Track the best validation Macro F1 score
best_val_f1 = 0.0

# Path to save best model checkpoint
best_model_path = FUSION_CHECKPOINT_DIR / "best_concat_fusion_model.pth"

for epoch in range(NUM_EPOCHS):
    # ---- Train for one epoch ----
    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    # ---- Evaluate on validation set ----
    val_loss, val_acc, val_f1, _, _ = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device
    )

    # Print epoch summary
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")

    # Save best model based on validation Macro F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model.")

Epoch 1/15
Train Loss: 0.0237 | Train Acc: 0.9930
Val Loss:   0.0002 | Val Acc:   1.0000 | Val Macro F1: 1.0000
Saved best model.
Epoch 2/15
Train Loss: 0.0130 | Train Acc: 0.9975
Val Loss:   0.0007 | Val Acc:   1.0000 | Val Macro F1: 1.0000
Epoch 3/15
Train Loss: 0.0128 | Train Acc: 0.9969
Val Loss:   0.0002 | Val Acc:   1.0000 | Val Macro F1: 1.0000
Epoch 4/15
Train Loss: 0.0123 | Train Acc: 0.9976
Val Loss:   0.0001 | Val Acc:   1.0000 | Val Macro F1: 1.0000
Epoch 5/15
Train Loss: 0.0154 | Train Acc: 0.9963
Val Loss:   0.0009 | Val Acc:   1.0000 | Val Macro F1: 1.0000
Epoch 6/15
Train Loss: 0.0109 | Train Acc: 0.9977
Val Loss:   0.0016 | Val Acc:   0.9995 | Val Macro F1: 0.9995
Epoch 7/15
Train Loss: 0.0079 | Train Acc: 0.9979
Val Loss:   0.0022 | Val Acc:   0.9995 | Val Macro F1: 0.9995
Epoch 8/15
Train Loss: 0.0145 | Train Acc: 0.9971
Val Loss:   0.0003 | Val Acc:   1.0000 | Val Macro F1: 1.0000
Epoch 9/15
Train Loss: 0.0096 | Train Acc: 0.9974
Val Loss:   0.0002 | Val Acc:   1.00

In [30]:
from sklearn.metrics import classification_report, confusion_matrix

# =========================
# Load best saved model
# =========================
model.load_state_dict(torch.load(best_model_path, map_location=device))

# =========================
# Final evaluation on test set
# =========================
test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc: {test_acc:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

Test Loss: 0.0082
Test Acc: 0.9977
Test Macro F1: 0.9977


In [31]:
 # =========================
# Detailed test metrics
# =========================
print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

Confusion Matrix:
[[718   2   0]
 [  3 717   0]
 [  0   0 720]]

Classification Report:
              precision    recall  f1-score   support

      supine       1.00      1.00      1.00       720
        left       1.00      1.00      1.00       720
       right       1.00      1.00      1.00       720

    accuracy                           1.00      2160
   macro avg       1.00      1.00      1.00      2160
weighted avg       1.00      1.00      1.00      2160

